# Vertical Slice — Objectives #1 (concentration) and #2 (skill calibration)

End-to-end analysis on the **top-100 resolved Polymarket markets** of the last 180 days, selected by total CLOB volume (`volumeClob` descending) — these are the markets where the integrity question is loudest because the money at stake is largest.

## What we want to answer

| Objective | Question | Answer comes from |
|---|---|---|
| **#1 Concentration** | Is Polymarket dominated by a handful of wallets, or is volume broadly distributed? | Gini, HHI, top-N share, Lorenz; computed per-market, per-negRisk-family, and platform-wide. |
| **#2 Skill calibration** | Do wallets that pay $X per share of an outcome actually win at rate X? Which wallets beat their own entry-price-implied probability? | Universe-level reliability curve + per-wallet Bayesian Beta posterior + realised dollar PnL cross-check. |

## Data pipeline

```
scripts/01_fetch_resolved_markets.py        # Gamma   /events + /markets   → markets.parquet, neg_risk_families.parquet
scripts/02_fetch_trades_and_holders.py      # Data API /trades + /holders  → trades/, holders/ (partitioned by condition_id)
```

This notebook then opens a DuckDB warehouse over those parquets via `intellifi.warehouse.open_warehouse()`, which registers views for `markets`, `trades`, `holders`, `neg_risk_families`, and a derived `winning_outcomes` (winner token_id per market, recovered from `outcomePrices`).

## Known data-quality limits (documented in `src/intellifi/data_api.py`)

* **Trades:** Data API `/trades` caps pagination at the most-recent **~4000 trades** per market (offset ≤ 3000, limit ≤ 1000 → ≤ 4000 reachable). For very high-volume markets this is a tail slice, not the full history. The subgraph layer in Phase 3 of the v2 spec would close this gap.
* **Holders:** snapshot, not time series. Reflects positions at the moment we queried, which for resolved markets is *after* resolution (so the holders weight ≈ realised exposure).
* **Identifiers:** every row carries `condition_id` (Data API key), `clob_token_ids` (CLOB key), and `slug` (human URL). These are never assumed interchangeable — see spec §5.2.

## How to read the rest of this notebook

The two objectives are reported at three levels of aggregation:

1. **Per market** (§1) — one row per `condition_id`. Tells us how concentrated/calibrated any single market is.
2. **Per negRisk family** (§1c) — multiple Yes/No markets that resolve mutually exclusively (e.g. "who wins the election"). Consolidated because trading one outcome reveals information about all of them; per-market figures would undercount cross-outcome flow.
3. **Platform-wide** (§1b) — every wallet's notional summed across all 100 markets. Tells us about Polymarket the venue, not any one market.

Each subsection states its rationale, methodology, and what the numbers would look like under the null (broad participation) versus the alternative (whale dominance).

In [ ]:
# Imports and warehouse bootstrap.
#   open_warehouse()   — DuckDB connection with views over the parquet dataset:
#                        markets, trades, holders, neg_risk_families, winning_outcomes.
#   build_bets_view()  — derives a `bets` view from `trades`: each trade is
#                        recast as a directional bet on the eventual outcome,
#                        with implied_p = price (BUY) or 1-price (SELL) and a
#                        boolean `won` flag computed against winning_outcomes.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

from intellifi.warehouse import open_warehouse
from intellifi.concentration import concentration_table, wallet_universe
from intellifi.skill import build_bets_view, calibration_by_bin, wallet_skill, wallet_pnl, SkillConfig

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 25)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110

con = open_warehouse()
build_bets_view(con)

# Sanity check on the warehouse: how many markets, trades, holders, wallets
# does the universe contain? n_resolved should equal n_markets here because
# Stage 1 only loaded resolved markets.
con.sql('''
    SELECT (SELECT COUNT(*) FROM markets)            AS n_markets,
           (SELECT COUNT(*) FROM trades)             AS n_trades,
           (SELECT COUNT(*) FROM holders)            AS n_holders_rows,
           (SELECT COUNT(DISTINCT proxy_wallet) FROM trades) AS n_unique_wallets_trading,
           (SELECT COUNT(*) FROM neg_risk_families)  AS n_family_rows,
           (SELECT COUNT(*) FROM winning_outcomes)   AS n_resolved
''').df()

## 1. Wallet concentration per market

### Rationale

Concentration metrics tell us whether a market's price was made by many independent participants or by a few large actors. A market with a high Gini and a high top-1 share is structurally easier to manipulate — a single wallet's flow can move the mid-price by tens of cents.

### Two weights side by side

We compute the same set of metrics under two different "weights" so the reader can see who is *active* vs who is *exposed*:

* `trade_notional` — sum of `price × size` per wallet across the market's lifetime (last ~4000 trades). Concentration of *activity*: who pushed the price around.
* `holdings_amount` — sum of conditional-token balances per wallet at the holders snapshot. Concentration of *positions*: who actually held shares when the market resolved.

The two diverge in a revealing way for market-makers (high trade notional, low residual holdings) vs directional bettors (low/medium trade notional, large residual holdings).

### Methodology (`src/intellifi/concentration.py`)

For each market we compute:

* **Gini** — area between the Lorenz curve and the equality line. `0` = perfect equality, `1` = one wallet owns everything.
* **HHI (Herfindahl-Hirschman Index)** — `Σ s_i²` where `s_i` is wallet `i`'s share. Sensitive to large holders specifically; doubles when one wallet's share doubles.
* **top1 / top5 / top10 / top25 share** — fraction of total weight held by the top N wallets, ranked descending. Intuitive but loses tail information.

All metrics are computed via a single DuckDB CTE that first computes per-wallet shares with a window function, then aggregates. See `_GINI_HHI_BODY` and `_GINI_HHI_AGG` in `concentration.py`.

### Expected ranges

* If Polymarket markets behaved like equity-market order books, we'd expect **Gini ≈ 0.7–0.85**, **top-10 share ≈ 0.25–0.40**. That would be "concentrated but not dominated".
* If markets are dominated by a small group of whales (the hypothesis we're testing for objective #1), we'd see **Gini > 0.9**, **top-10 share > 0.5**, especially on holdings.

In [ ]:
# Compute concentration metrics per market under both weight schemes,
# then average across all 100 markets to get a cross-market headline number.
# n_wallets   — distinct wallets contributing positive weight
# gini, hhi   — defined above (Σ s_i² for HHI)
# topN_share  — fraction of total weight in the top N wallets
conc_t = concentration_table(con, weight='trade_notional', group='market').df()
conc_h = concentration_table(con, weight='holdings_amount', group='market').df()

summary = pd.DataFrame({
    'trade_notional': conc_t[['n_wallets', 'gini', 'hhi', 'top1_share', 'top5_share', 'top10_share', 'top25_share']].mean(),
    'holdings_amount': conc_h[['n_wallets', 'gini', 'hhi', 'top1_share', 'top5_share', 'top10_share', 'top25_share']].mean(),
})
print('Cross-market means:')
summary.round(3)

In [ ]:
# Distribution of per-market Gini coefficients. The bulk of the mass should
# sit far to the right (>0.9) if the platform is whale-dominated; a unimodal
# distribution near 0.5–0.7 would indicate broad participation.
# We expect the holdings distribution to be slightly less concentrated than
# trade notional because market-makers churn notional without retaining
# positions — their footprint shows up on trades but not on holdings.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, df, title in [(axes[0], conc_t, 'Trade notional'), (axes[1], conc_h, 'Holdings at resolution')]:
    ax.hist(df['gini'].dropna(), bins=25, color='steelblue', edgecolor='black', alpha=0.85)
    ax.axvline(df['gini'].mean(), color='crimson', linestyle='--', label=f"mean = {df['gini'].mean():.3f}")
    ax.set_title(f'Gini distribution — {title}')
    ax.set_xlabel('Gini coefficient')
    ax.set_ylabel('Markets')
    ax.legend()
plt.tight_layout()

In [ ]:
# Bar comparison: how much of the average market does the top-1, top-5,
# top-10, top-25 wallets capture? The two bars per group make the contrast
# between trade-activity concentration and position concentration visible.
# Reading: if top10 (orange) on holdings is > 0.6, then 10 wallets hold
# more shares at resolution than the other 600+ holders combined.
shares = pd.DataFrame({
    'top1':  [conc_t['top1_share'].mean(),  conc_h['top1_share'].mean()],
    'top5':  [conc_t['top5_share'].mean(),  conc_h['top5_share'].mean()],
    'top10': [conc_t['top10_share'].mean(), conc_h['top10_share'].mean()],
    'top25': [conc_t['top25_share'].mean(), conc_h['top25_share'].mean()],
}, index=['trade_notional', 'holdings_amount']).T

ax = shares.plot(kind='bar', figsize=(9, 4), color=['steelblue', 'darkorange'], edgecolor='black')
ax.set_ylabel('Average share of total weight')
ax.set_title('Top-N wallets capture (averaged across 100 markets)')
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', label_type='edge', fontsize=8)
plt.tight_layout()

In [ ]:
# Lorenz curves for three reference markets at high / mid / low volume of
# the universe. The Lorenz plots cumulative share of total weight (y-axis)
# against cumulative share of wallets sorted small-to-large (x-axis).
# The dashed diagonal is perfect equality; the further a curve bows below
# the diagonal, the more concentrated the market.
# Picking a high-vol, mid-vol, and low-vol market lets the reader judge
# whether concentration is uniform across the volume distribution.
def lorenz(weights):
    w = np.sort(np.asarray(weights, dtype=float))
    w = w[w > 0]
    if w.size == 0:
        return np.array([0, 1]), np.array([0, 1])
    cum = np.cumsum(w) / w.sum()
    x = np.linspace(0, 1, w.size + 1)
    y = np.concatenate(([0], cum))
    return x, y

picks = conc_t.iloc[[0, 25, 75]]  # high-, mid-, low-volume market by sort order
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='perfect equality')
for _, row in picks.iterrows():
    cid = row['group_key']
    weights = con.execute(
        '''SELECT SUM(notional_usdc) AS w FROM trades
           WHERE condition_id = ? AND proxy_wallet IS NOT NULL AND notional_usdc > 0
           GROUP BY proxy_wallet''', [cid]).fetchall()
    x, y = lorenz([w[0] for w in weights])
    ax.plot(x, y, label=f"{row['slug'][:40]} (G={row['gini']:.2f}, vol=${row['volume_clob']/1e6:.1f}M)")
ax.set_xlabel('Cumulative share of wallets (sorted small → large)')
ax.set_ylabel('Cumulative share of trade notional')
ax.set_title('Lorenz curves — trade-notional concentration')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()

In [ ]:
# Does concentration scale with market size? Plot top-10 wallet share vs
# `volume_clob` (log) and colour by negRisk flag. Two hypotheses:
#   (a) larger markets are more competitive → top-10 share decreases with volume
#   (b) larger markets attract proportionally more whales → top-10 share is
#       independent of volume (flat scatter)
# Empirically the cloud is approximately flat across two orders of magnitude
# of volume, consistent with (b): concentration is a structural feature of
# Polymarket, not a small-market artefact.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, df, title in [(axes[0], conc_t, 'Trade notional'), (axes[1], conc_h, 'Holdings')]:
    ax.scatter(df['volume_clob'] / 1e6, df['top10_share'],
               c=df['neg_risk'].astype(int).map({0: 'steelblue', 1: 'darkorange'}),
               s=30, alpha=0.75, edgecolor='black', linewidth=0.3)
    ax.set_xscale('log')
    ax.set_xlabel('Market volume_clob (USDC, log)')
    ax.set_ylabel('Top-10 wallet share')
    ax.set_title(f'{title} — concentration vs. market size')
    for c, lbl in [('steelblue', 'non-negRisk'), ('darkorange', 'negRisk')]:
        ax.scatter([], [], c=c, label=lbl, edgecolor='black', linewidth=0.3)
    ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()

## 1b. Cross-market concentration (the whole universe in one view)

### Rationale

Per-market concentration can be high while platform-wide concentration is low — that would happen if different sets of whales dominate different markets. Conversely, if the *same* wallets dominate market after market, platform-wide Gini ends up higher than the per-market mean. This level of aggregation directly answers "is Polymarket centralised?" and is the diagnostic for the entity-resolution work in notebook 2.

### Methodology

`intellifi.concentration.wallet_universe()` sums every wallet's weight across all markets in the dataset and returns a sorted DataFrame. We then re-apply the same Gini and top-N formulas at the universe level.

Two weights again:

* **trade_notional** sums `Σ price × size` per wallet across all 100 markets.
* **holdings_amount** sums conditional-token balances per wallet across all holders snapshots.

### What to expect

* **If markets are run by the same crew everywhere**: universe Gini ≥ mean per-market Gini, top-100 platform share ≈ top-10 single-market share.
* **If each market has its own ecosystem**: universe Gini < mean per-market Gini because aggregation averages out concentration.

In [ ]:
# Aggregate weights across all 100 markets and report platform-level
# headline numbers. ~139k unique wallets traded; ~52k still held positions
# at resolution. Total trade notional is ~$0.44B (≈ 4-month subsample of
# the top-100 markets). gini_np() is the standard equal-step Lorenz Gini.
wu_t = wallet_universe(con, weight='trade_notional').df()
wu_h = wallet_universe(con, weight='holdings_amount').df()

print(f"Trade universe:    {len(wu_t):>7,d} wallets, total notional ${wu_t['total_notional'].sum()/1e9:.2f}B")
print(f"Holdings universe: {len(wu_h):>7,d} wallets, total amount   ${wu_h['total_amount'].sum()/1e6:.1f}M shares")

def gini_np(w):
    # Standard Gini for a non-negative weight vector. Sorted ascending,
    # then G = (2 * Σ i*w_i) / (n * Σ w_i) - (n+1)/n.
    w = np.sort(np.asarray(w, dtype=float))
    w = w[w > 0]
    n = w.size
    if n < 2:
        return float('nan')
    return (2 * (np.arange(1, n + 1) * w).sum()) / (n * w.sum()) - (n + 1) / n

topn = [10, 100, 1000]
rows = []
for name, df, col in [('trade_notional', wu_t, 'total_notional'),
                      ('holdings_amount', wu_h, 'total_amount')]:
    total = df[col].sum()
    row = {'metric': name, 'gini': gini_np(df[col].values)}
    for n in topn:
        row[f'top{n}_share'] = df[col].head(n).sum() / total
    rows.append(row)
pd.DataFrame(rows).round(4)

In [ ]:
# Platform-wide Lorenz curve. Both weights bow heavily below the equality
# diagonal: the bottom ~90% of wallets account for <10% of total weight,
# while the top 100 hold ~50% of all trade notional. This is the headline
# image for "objective #1, answered".
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='perfect equality')
for name, df, col, c in [('Trade notional', wu_t, 'total_notional', 'steelblue'),
                          ('Holdings amount', wu_h, 'total_amount', 'darkorange')]:
    x, y = lorenz(df[col].values)
    ax.plot(x, y, color=c, label=f"{name} (G={gini_np(df[col].values):.3f})")
ax.set_xlabel('Cumulative share of wallets')
ax.set_ylabel('Cumulative share of weight')
ax.set_title('Platform-wide concentration (Lorenz)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()

## 1c. negRisk family-level concentration

### Why bother

A **negRisk family** is a set of related Yes/No markets that resolve mutually exclusively — for example, a "who wins the election" event becomes one Yes/No market per candidate, all linked by a shared `event_id`. A wallet that buys YES on candidate A and YES on candidate B has effectively expressed two related views, and per-v2-spec §4.4 we must consolidate flow across the *whole family* before measuring concentration or signing pressure. Looking at the candidate-A market in isolation undercounts the wallet's footprint.

### Methodology

`concentration_table(con, group='family')` joins `trades` to `neg_risk_families` on `event_id`, then computes Gini/HHI/top-N per family rather than per market. Markets not in a negRisk family are passed through as singleton groups so the report contains every market exactly once.

### What to expect

* Family-level Gini should be **similar to or slightly lower** than market-level Gini — aggregation across outcomes typically averages out small fluctuations.
* Top-10 share should also be similar; if it falls dramatically when grouping by family, that means different wallets dominate different outcomes within the same event (broadly distributed event-level participation), which is the *less* concerning case.
* If family-level concentration is *higher* than market-level, the same wallets dominate every outcome of a multi-outcome event — i.e. a single entity is taking positions across the entire event. That is the more concerning case for market integrity.

In [ ]:
# Family-level concentration table. The volume_clob column is *per family*
# (not per market) — that's why the top rows all show the same large number:
# they belong to the same mega-event, and slug is NaN at the family level
# because there is no single "family slug" — each family has multiple
# member markets each with their own slug.
fam_conc = concentration_table(con, weight='trade_notional', group='family').df()
print(f"groups (negRisk families + lone non-negRisk markets): {len(fam_conc)}")
fam_conc.sort_values('volume_clob', ascending=False).head(10)[
    ['slug', 'volume_clob', 'n_wallets', 'gini', 'top10_share']
]

## 2. Skill calibration — universe reliability curve

### Rationale

The price of a Polymarket outcome token is the market's implied probability that the outcome will resolve YES. If the market is well-calibrated, a token that trades at $0.30 should win 30% of the time, on average, across all such trades. The empirical relationship between the bucketed implied probability and the realised hit rate is the **reliability curve**.

This is the foundational diagnostic for "is Polymarket informationally efficient at the price level?" — it underpins the skill-gap and PnL analyses that follow. If the curve already sits on the diagonal everywhere, there is *no* edge to find. If it deviates systematically in some price range, every wallet trading in that range is either consistently right or consistently wrong, and we can rank them.

### Methodology (`src/intellifi/skill.py::calibration_by_bin`)

1. From the `bets` view (built earlier from `trades` + `winning_outcomes`), each row is a directional bet with an `implied_p` and a boolean `won`.
2. Bin every bet by `implied_p` into 20 equal-width bins on `[0, 1]`.
3. For each bin compute: `n_trades`, `mean_implied_p`, `realised_hit_rate` (= average of `won`), `total_weight`, and the gap.
4. Run twice: once with `weight='size'` (weighted by share count — what dollar flow actually paid), once with `weight='count'` (one vote per trade — sensitive to retail).

### What to expect

* **Diagonal alignment is the null.** A well-calibrated market sits on `y = x` everywhere.
* **Longshot bias** is the canonical departure: in cheap bins (p < 0.3) realised hit rate is below the implied probability — bettors over-pay for unlikely outcomes. This is documented in horse-racing and sports-book literature for decades, and is the textbook prediction here.
* **Favourite-longshot symmetry** is the opposite departure: in the p > 0.7 bins the realised hit rate exceeds the implied — favourites are under-priced. Less common in calibrated markets.
* **Middle bins (0.4–0.6)** carry less weight and are inherently noisy because (a) those prices reflect genuinely uncertain markets and (b) by construction there is less mass there in our weighted sample.

In [ ]:
# Reliability curve in two views:
#   blue dots — size-weighted (every share counts) — what dollar flow paid
#   red x'es  — count-weighted (every trade counts) — sensitive to retail
# Point size ∝ √n_trades to make small-sample bins visually quieter.
# Reading: dots ABOVE the diagonal => prices systematically under-state the
# eventual probability (e.g. favourites at p≈0.75 actually win ~99%).
# Dots BELOW the diagonal => prices over-state (longshot bias).
cal = calibration_by_bin(con, n_bins=20, weight='size').df()
cal_count = calibration_by_bin(con, n_bins=20, weight='count').df()

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='perfectly calibrated')
sizes = cal['n_trades'].clip(lower=5)
ax.scatter(cal['mean_implied_p'], cal['realised_hit_rate'],
           s=np.sqrt(sizes) * 4, alpha=0.75, color='steelblue', edgecolor='black',
           label='size-weighted (point size ∝ √n_trades)')
ax.scatter(cal_count['mean_implied_p'], cal_count['realised_hit_rate'],
           s=15, alpha=0.6, color='crimson', marker='x', label='count-weighted')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Mean implied probability in bin')
ax.set_ylabel('Realised hit rate')
ax.set_title('Reliability curve — 20 bins')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Tabular view of the same data — handy for reading bin-by-bin gaps.
# calibration_gap = realised_hit_rate - mean_implied_p:
#   negative gap => buyers overpaid for that probability range (longshot bias)
#   positive gap => buyers under-paid (favourites systematically win)
# Note that bins 0 (p≈0.003) and 19 (p≈0.997) carry >90% of all trades:
# most of the dollar flow happens after a market has effectively decided.
print('Universe reliability (size-weighted, 20 bins):')
cal.round(4)

### Interpretation

* **Longshot bias (p < 0.05–0.30).** Realised hit rate is *even smaller* than the mean implied probability — buyers are over-paying for cheap longshots. Bin 0 contains a huge mass of trades at p≈0.003 that resolved to 0; bins 1–8 also sit visibly below the diagonal.
* **Favourite mispricing in the middle-favourite range (p ≈ 0.6–0.8).** Realised hit rate sits *above* the implied — these tokens are systematically under-priced. Bin 12 (p≈0.63) actually realised 91.6% — that's a 28-percentage-point favourite-mispricing gap.
* **Tail calibration (p > 0.9).** Almost perfectly on the diagonal: by the time a token trades at $0.93+, it has decided and the market reflects it accurately.
* **Middle bins (p ≈ 0.3–0.5).** Wide gaps (some up to 37pp) but with very few observations — these bins are dominated by noise and a handful of specific markets.

**Takeaway:** Polymarket exhibits the classic longshot bias on the cheap end, plus a mid-favourite under-pricing region. Both are systematic departures from calibration — and they are exactly the price ranges where wallets that *don't* exhibit those biases will accumulate a positive calibration gap. That is what we measure next.

## 2b. Per-wallet Bayesian skill posterior

### Rationale

The universe-level reliability curve answers "is Polymarket calibrated on average". This subsection asks the per-wallet version: **which individual wallets win more often than the prices they paid implied?** A wallet with a persistently positive `realised_hit_rate − mean_implied_p` gap is either skilled, informed, or lucky in a small sample. The Bayesian framing separates the three.

### Why Bayesian (and not just `hits / trades`)

A wallet with 4 wins out of 5 trades looks like a 80% hit rate, but the sample is too small to compete with a wallet that has 70 wins out of 100. The Beta posterior collapses to the empirical rate as `n → ∞` and shrinks toward the prior at small `n`, giving us a single comparable statistic across wallets with very different sample sizes.

### Methodology (`src/intellifi/skill.py::wallet_skill`)

For each wallet with at least `min_trades=20` size-weighted trades:

1. Compute `mean_implied_p` — the wallet's average entry-price-implied probability across its bets.
2. Set a **Beta prior** `Beta(α₀, β₀)` with `α₀ + β₀ = prior_strength = 4` (weak — overwhelmed by any meaningful sample), centred on the wallet's own `mean_implied_p`:
   * `α₀ = prior_strength × mean_implied_p`
   * `β₀ = prior_strength × (1 − mean_implied_p)`
   The prior says: "if I knew nothing, I'd expect this wallet to win at the rate the market priced its bets, on average."
3. Update with observed `(wins, losses)` weighted by trade `size`:
   * `α = α₀ + Σ size × won`
   * `β = β₀ + Σ size × (1 − won)`
4. **Calibration gap** = `posterior_mean − mean_implied_p` = `α/(α+β) − mean_implied_p`. Positive means the wallet wins more often than the prices it paid imply.
5. Report a 95% credible interval `[ci_low, ci_high]` from the Beta quantiles.

The prior centring on the wallet's own implied probability is important: it makes the *null hypothesis* "this wallet wins at the rate it paid for" instead of "this wallet wins 50% of the time". The calibration gap then measures the wallet's edge above the rate it bought in at, not above some arbitrary 50/50 baseline.

### What to expect

* If skill is concentrated in a small group, the histogram of `calibration_gap` should be a long-tailed right-skewed distribution (most wallets near 0, a fat right tail of skilled wallets).
* If "skill" is mostly noise, the histogram should look symmetric around 0 — for every wallet with a +0.2 gap, there's roughly one with a −0.2 gap.
* Survivorship and selection effects are real: a wallet that only takes one big winning bet ends up in the right tail. The "n_trades vs gap" scatter below tests for this.

In [ ]:
# Per-wallet skill posteriors, sorted by calibration_gap descending.
# Columns:
#   n_trades, n_markets    — sample size and breadth
#   mean_implied_p         — average entry-price-implied probability (prior centre)
#   realised_hit_rate      — empirical fraction won, size-weighted
#   posterior_mean_hit_rate— Beta posterior mean after observed wins/losses
#   calibration_gap        — posterior_mean - mean_implied_p; positive = edge
#   ci_low, ci_high        — 95% credible interval (frequentist analogue: a confidence interval)
#   total_notional         — dollar size of all bets (helps distinguish whales from retail)
ws = wallet_skill(con, cfg=SkillConfig(min_trades=20, prior_strength=4.0)).df()
print(f'wallets with >=20 size-weighted trades: {len(ws)}')
ws[['proxy_wallet', 'n_trades', 'n_markets', 'mean_implied_p',
    'realised_hit_rate', 'posterior_mean_hit_rate', 'calibration_gap',
    'ci_low', 'ci_high', 'total_notional']].head(15)

In [ ]:
# Distribution of per-wallet calibration gaps. A symmetric distribution
# centred on 0 would say "skill is noise". A right-skewed distribution with
# a fat tail of large positive gaps says "a meaningful minority has edge".
# The vertical lines mark zero (no edge) and the sample mean (the population
# average wallet has a small positive gap — partly genuine skill, partly the
# longshot-bias undercurrent showing through).
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ws['calibration_gap'], bins=60, color='steelblue', edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', linestyle='--', alpha=0.6, label='no edge')
ax.axvline(ws['calibration_gap'].mean(), color='crimson', linestyle='--',
           label=f"mean = {ws['calibration_gap'].mean():.3f}")
ax.set_xlabel('Calibration gap (posterior_mean_hit_rate - mean_implied_p)')
ax.set_ylabel('Wallets')
ax.set_title(f'Wallet calibration gap distribution (n={len(ws)} wallets)')
ax.legend()
plt.tight_layout()

In [ ]:
# Sanity test for small-sample inflation. If the top calibration gaps were
# all driven by wallets with 20-50 trades and tiny notionals, the right tail
# would just be noise. By colouring points by log10(total_notional), we can
# spot whether the high-gap wallets are also financially serious traders.
# A genuine skill cluster appears as bright (high-notional) dots at high
# n_trades with positive gaps; small-sample artefacts appear as dim
# (low-notional) dots clustered on the left.
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(ws['n_trades'], ws['calibration_gap'],
                c=np.log10(ws['total_notional'].clip(lower=1)), cmap='viridis',
                s=15, alpha=0.6, edgecolor='none')
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('n_trades (resolved markets, log)')
ax.set_ylabel('calibration gap')
cbar = plt.colorbar(sc); cbar.set_label('log10(total_notional)')
ax.set_title('Are positive-gap wallets small-sample artefacts? (color = $ size)')
plt.tight_layout()

In [ ]:
# For the five wallets with the largest calibration gaps, plot the Beta
# posterior density over hit rate. The narrower the peak, the tighter the
# evidence; a peak near 1.0 with a long left tail says "almost certainly
# wins most of its trades but the prior pulls slightly left". The label
# also shows the wallet's own mean implied probability (meanIP) — the gap
# between the peak and meanIP is the visualised calibration_gap.
top5 = ws.head(5)
fig, ax = plt.subplots(figsize=(9, 5))
xs = np.linspace(0, 1, 400)
for _, row in top5.iterrows():
    a, b = float(row['posterior_alpha']), float(row['posterior_beta'])
    ax.plot(xs, beta_dist.pdf(xs, a, b),
            label=f"{row['proxy_wallet'][:10]}…  n={int(row['n_trades'])}  E[p]={a/(a+b):.2f}  meanIP={row['mean_implied_p']:.2f}")
ax.set_xlabel('hit-rate posterior')
ax.set_ylabel('density')
ax.set_title('Beta posterior over hit rate — top-5 calibrated wallets')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()

## 2c. Realised dollar PnL

### Rationale

The Bayesian calibration gap is a *probability-space* statistic — it answers "did this wallet pick winners better than the prices implied?". Realised dollar PnL is the *money-space* check: "did this wallet walk away with USDC?". The two should agree at the top of the ranking; if they don't, that's an alarm.

A wallet can have a positive calibration gap but lose money — for instance by trading tiny size on cheap winners and large size on expensive losers. Conversely, a wallet can have a near-zero calibration gap and still make money if it's a market maker capturing spread without taking directional risk.

### Methodology (`src/intellifi/skill.py::wallet_pnl`)

For each trade we approximate per-share notional PnL:

* **BUY** of `size` shares at `price`: pays `size × price` USDC; receives `size × outcome` USDC at resolution (`outcome ∈ {0, 1}`). PnL = `size × (outcome − price)`.
* **SELL** is the mirror: receives `size × price`; loses `size × outcome`. PnL = `size × (price − outcome)`.

Summing across all trades for a wallet gives realised notional PnL. The aggregate of *all* wallets should be approximately zero (every dollar won = dollar lost) up to AMM/fee leakage; we print this as a sanity check.

### Caveats

* **Per-share, not per-position.** This does not net cross-market exposures or distinguish realised vs. unrealised. A wallet that opened and closed a position before resolution still shows up here at the price difference; one that held to resolution shows up at the full settlement value.
* **No fee model.** Polymarket on-chain trades pay gas + small protocol/exchange fees that we don't subtract here. The full-fee picture is run inside the backtest (notebook 3) where it matters for strategy returns.
* **Currency.** Quoted in `USDC` units because that's the unit of `notional_usdc` from the Data API.

In [ ]:
# Realised PnL leaderboard. The aggregate sum is a sanity check: it should
# be close to zero (zero-sum minus fees). The ~$3M negative residual is the
# expected drag from fees + AMM impact accumulated across the universe.
# Top wallets here will be compared against the skill leaderboard above —
# we want to see ranking agreement at the top.
pnl = wallet_pnl(con).df()
print(f'wallets with realised PnL: {len(pnl)}')
print(f'aggregate sum (should ~zero if every $ won = $ lost): ${pnl["realised_pnl_usdc"].sum():,.0f}')
pnl.head(15)

In [ ]:
# Joint plot — calibration gap (x) vs realised PnL (y, symmetric log).
# Expected if the skill score is meaningful: positive correlation, with the
# upper-right quadrant populated and the lower-right empty.
# Dots in the upper-LEFT quadrant are the alarm: wallets with NEGATIVE
# calibration gaps that still made money. They're either (a) making PnL
# from spread / market-making, not from directional picks, or (b) trading
# heavily in mispriced bins where even a wrong-direction average position
# can be profitable.
joined = ws.merge(pnl, on='proxy_wallet', how='inner', suffixes=('', '_pnl'))
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(joined['calibration_gap'], joined['realised_pnl_usdc'],
           s=np.sqrt(joined['n_trades']) * 5, alpha=0.5, edgecolor='none', color='steelblue')
ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.set_yscale('symlog', linthresh=100)
ax.set_xlabel('Calibration gap')
ax.set_ylabel('Realised PnL (USDC, symlog)')
ax.set_title('Calibration gap vs dollar PnL (point ∝ √n_trades)')
plt.tight_layout()

## Summary — objectives #1 and #2

### What we found

**Concentration (objective #1).** Very high.
* Mean per-market Gini: **0.94** on trade notional, **0.91** on holdings.
* Top-10 wallets capture ~53% of trade volume and ~66% of holdings on a typical market.
* Platform-wide across all 100 markets, the Lorenz is steeper still: 100 wallets account for ~50% of all trading notional.
* Concentration is roughly invariant to market size — it is structural, not a small-market artefact.

**Calibration (objective #2).** The platform is approximately calibrated only at the extremes.
* **Longshot bias** on the cheap end (p < 0.30): realised hit rate is below the implied probability — buyers systematically overpay.
* **Favourite under-pricing** in the middle-favourite range (p ≈ 0.6–0.8): realised hit rate is *above* the implied — these tokens are systematically cheap.
* Tail bins (p > 0.9) sit on the calibration diagonal.
* Per-wallet, the calibration-gap distribution has a fat right tail. The top wallets in that tail are also large by `n_trades` and `total_notional`, so the right tail is not a small-sample artefact — these wallets are the candidates for the "skill / informedness / coordination" investigations in notebooks 2 and 3.

### What this means for the manipulation lens

* Structural whale dominance plus a measurable longshot bias gives a small set of wallets both the *motive* (mispriced prices to exploit) and the *means* (size to move them) for the kind of activity v2 spec §20 enumerates as suspicious.
* The next questions — "are the top wallets one entity or several?" (notebook 2) and "do these wallets actually have predictive edge or just survivorship?" (notebook 3) — directly follow from this slice.

### Caveats baked into these numbers

* Trades view is capped at the most-recent ~4000 trades per market (Data API limit). High-volume markets are under-sampled in their early phase; the on-chain subgraph layer (Phase 3 of the spec) would close this gap.
* PnL is per-share notional; it does not net cross-market positions or distinguish realised vs unrealised.
* The wallet identity is the **proxy wallet only** — Sybils / multi-account same-entity are not separated. That is precisely the objective the next notebook tackles.